<button data-commandLinker-command="progressivis:cleanup_and_run" data-commandlinker-args='{"index": 1}' href="#" class='progressivis-cleanup-and-run-btn'>Run ProgressiVis</button>

In [ ]:
from ipyprogressivis.widgets.chaining.constructor import Constructor
from ipyprogressivis.widgets.chaining.utils import create_root, get_header
from ipyprogressivis.widgets.chaining.custom import *
# ***************************************************************************************
# WARNING: This cell must only be executed using the 'Run ProgressiVis' button above.
# Do not execute it in any other way, as the result will not be as expected.
# For the same reason do not copy/paste the contents of this cell to execute it elsewhere
# ***************************************************************************************
header = get_header()
display(header.talker)
display(header.backup)
_ = header.constructor
with header.modules_out:
    display(header.board)
with header.widgets_out:
    display(header.manager)
header.talker.labcommand("notebook:hide-cell-code")
%reload_ext ipyprogressivis.magics
create_root(header.backup)

Talker()

BackupWidget()

## root

In [1]:
# do not run this cell
display(header.constructor)
header.constructor.start_scheduler()
header.talker.labcommand('notebook:hide-cell-code')

Constructor(children=(IntProgress(value=0, description='Starting ProgressiVis ...', max=2, style=ProgressStyle…

Starting scheduler
# Scheduler added module(s): ['sink_1', 'variable_1']
# Scheduler added module(s): ['data_shape_1', 'sink_3']
# Scheduler deleted module(s): {'data_shape_1', 'sink_3'}
# Scheduler added module(s): ['data_shape_1', 'sink_3']
# Scheduler deleted module(s): {'data_shape_1', 'sink_3'}
# Scheduler added module(s): ['data_shape_1', 'sink_3']
# Scheduler deleted module(s): {'data_shape_1', 'sink_3'}


## CSV loader

In [2]:
Constructor.widget('CSV loader', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

In [3]:
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from datetime import datetime
from collections import defaultdict

@register_snippet
def progressive_escalation_dashboard(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_escalation_dashboard, "_state"):

        fig = go.FigureWidget()

        state = {
            "cursor": 0,
            "incident_seen": set(),
            "counts": defaultdict(int),
            "totals": defaultdict(float),
            "means": defaultdict(list)
        }

        progressive_escalation_dashboard._state = state

        progress_label = Label(value="Processing: 0%")

        with out:
            display(progress_label)
            display(fig)

        def update_plot():

            fig.data = []

            # 🎨 ColorBrewer Sequential Blues (4-class)
            colors = {
                "0": "#eff3ff",   # Lightest (Low escalation)
                "1": "#bdd7e7",
                "2": "#6baed6",
                "3+": "#2171b5"   # Darkest (High escalation)
            }

            for group in sorted(state["means"].keys()):
                fig.add_trace(go.Scatter(
                    y=state["means"][group],
                    mode="lines",
                    name=f"{group} Reassign",
                    line=dict(color=colors.get(group, "#999999"), width=3)
                ))

            fig.update_layout(
                title="Escalation Impact on Resolution Time",
                xaxis_title="Processed Closed Incidents",
                yaxis_title="Mean Resolution Time (Hours)",
                height=550,
                legend_title="Reassignment Count"
            )

        async def process_rows():

            batch_size = 100

            while state["cursor"] < table.nrow:

                start_i = state["cursor"]
                end_i = min(start_i + batch_size, table.nrow)

                for i in range(start_i, end_i):

                    row = table.row(i)

                    if row.get("incident_state") != "Closed":
                        continue

                    inc = row["number"]

                    if inc in state["incident_seen"]:
                        continue

                    try:
                        start = datetime.strptime(row["opened_at"], "%d/%m/%Y %H:%M")
                        end = datetime.strptime(row["closed_at"], "%d/%m/%Y %H:%M")
                    except:
                        continue

                    duration = (end - start).total_seconds() / 3600
                    esc = int(row.get("reassignment_count", 0))
                    esc_group = str(esc) if esc < 3 else "3+"

                    state["incident_seen"].add(inc)

                    state["counts"][esc_group] += 1
                    state["totals"][esc_group] += duration

                    mean = state["totals"][esc_group] / state["counts"][esc_group]
                    state["means"][esc_group].append(mean)

                state["cursor"] = end_i

                percent = round((state["cursor"] / table.nrow) * 100, 1)
                remaining = table.nrow - state["cursor"]

                progress_label.value = (
                    f"Processed: {percent}% | Used Rows: {state['cursor']} | Remaining: {remaining}"
                )

                update_plot()
                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )


## Snippet

In [4]:
Constructor.widget('Snippet', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

In [5]:
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from datetime import datetime
from collections import defaultdict
import re

@register_snippet
def progressive_priority_dashboard(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_priority_dashboard, "_state"):

        fig = go.FigureWidget()

        state = {
            "cursor": 0,
            "incident_seen": set(),
            "counts": defaultdict(int),
            "totals": defaultdict(float),
            "means": defaultdict(list)
        }

        progressive_priority_dashboard._state = state

        progress_label = Label(value="Processing: 0%")

        with out:
            display(progress_label)
            display(fig)

        def update_plot():

            fig.data = []

            # 🎨 ColorBrewer Sequential Oranges (5-class)
            colors = {
                1: "#a63603",  # Darkest (Highest Priority)
                2: "#e6550d",
                3: "#fd8d3c",
                4: "#fdbe85",
                5: "#feedde"   # Lightest (Lowest Priority)
            }

            for group in sorted(state["means"].keys()):
                fig.add_trace(go.Scatter(
                    y=state["means"][group],
                    mode="lines",
                    name=f"Priority {group}",
                    line=dict(color=colors.get(group, "#999999"), width=3)
                ))

            fig.update_layout(
                title="Priority Impact on Resolution Time",
                xaxis_title="Processed Closed Incidents",
                yaxis_title="Mean Resolution Time (Hours)",
                height=550,
                legend_title="Priority Level"
            )

        async def process_rows():

            batch_size = 100

            while state["cursor"] < table.nrow:

                start_i = state["cursor"]
                end_i = min(start_i + batch_size, table.nrow)

                for i in range(start_i, end_i):

                    row = table.row(i)

                    if row.get("incident_state") != "Closed":
                        continue

                    inc = row["number"]

                    if inc in state["incident_seen"]:
                        continue

                    try:
                        start = datetime.strptime(row["opened_at"], "%d/%m/%Y %H:%M")
                        end = datetime.strptime(row["closed_at"], "%d/%m/%Y %H:%M")
                    except:
                        continue

                    duration = (end - start).total_seconds() / 3600

                    pr_raw = str(row.get("priority", "3"))
                    m = re.search(r"\d+", pr_raw)
                    pr = int(m.group(0)) if m else 3

                    state["incident_seen"].add(inc)

                    state["counts"][pr] += 1
                    state["totals"][pr] += duration

                    mean = state["totals"][pr] / state["counts"][pr]
                    state["means"][pr].append(mean)

                state["cursor"] = end_i

                percent = round((state["cursor"] / table.nrow) * 100, 1)
                remaining = table.nrow - state["cursor"]

                progress_label.value = (
                    f"Processed: {percent}% | Used Rows: {state['cursor']} | Remaining: {remaining}"
                )

                update_plot()
                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )


## Snippet[1]

In [6]:
Constructor.widget('Snippet', 1)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Snippet[2]

In [10]:
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, Label, Dropdown, VBox
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import re

@register_snippet
def progressive_sla_dashboard(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_sla_dashboard, "_state"):

        fig = go.FigureWidget()

        group_dropdown = Dropdown(
            options=["Priority", "Escalation", "Category"],
            value="Priority",
            description="Group By:"
        )

        state = {
            "cursor": 0,
            "incident_seen": set(),
            "group_type": "Priority",
            "total_counts": defaultdict(int),
            "breach_counts": defaultdict(int),
            "breach_rates": defaultdict(list)
        }

        progressive_sla_dashboard._state = state

        progress_label = Label(value="Processing: 0%")

        with out:
            display(VBox([group_dropdown, progress_label]))
            display(fig)

        def reset_state():
            state["total_counts"].clear()
            state["breach_counts"].clear()
            state["breach_rates"].clear()
            fig.data = []

        def get_group(row):

            if state["group_type"] == "Priority":
                pr_raw = str(row.get("priority", "3"))
                m = re.search(r"\d+", pr_raw)
                return int(m.group(0)) if m else 3

            elif state["group_type"] == "Escalation":
                esc = int(row.get("reassignment_count", 0))
                return str(esc) if esc < 3 else "3+"

            elif state["group_type"] == "Category":
                return row.get("category", "Unknown")

        def get_color(group):

            # 🎨 Sequential Oranges for Priority
            if state["group_type"] == "Priority":
                oranges = {
                    1: "#a63603",
                    2: "#e6550d",
                    3: "#fd8d3c",
                    4: "#fdbe85",
                    5: "#feedde"
                }
                return oranges.get(group, "#999999")

            # 🎨 Sequential Blues for Escalation
            elif state["group_type"] == "Escalation":
                blues = {
                    "0": "#eff3ff",
                    "1": "#bdd7e7",
                    "2": "#6baed6",
                    "3+": "#2171b5"
                }
                return blues.get(group, "#999999")

            # 🎨 Qualitative Set2 for Category
            elif state["group_type"] == "Category":
                set2 = [
                    "#66c2a5", "#fc8d62", "#8da0cb",
                    "#e78ac3", "#a6d854", "#ffd92f",
                    "#e5c494", "#b3b3b3"
                ]
                return set2[hash(group) % len(set2)]

        def update_plot():

            fig.data = []

            for group in sorted(state["breach_rates"].keys()):
                fig.add_trace(go.Scatter(
                    y=state["breach_rates"][group],
                    mode="lines",
                    name=str(group),
                    line=dict(color=get_color(group), width=3)
                ))

            fig.update_layout(
                title=f"SLA Breach Rate by {state['group_type']}",
                xaxis_title="Processed Closed Incidents",
                yaxis_title="SLA Breach Rate (%)",
                height=550,
                legend_title=state["group_type"]
            )

        def on_group_change(change):
            state["group_type"] = change["new"]
            state["incident_seen"].clear()
            state["cursor"] = 0
            reset_state()
            aio.create_task(process_rows())

        group_dropdown.observe(on_group_change, names="value")

        async def process_rows():

            batch_size = 100

            while state["cursor"] < table.nrow:

                start_i = state["cursor"]
                end_i = min(start_i + batch_size, table.nrow)

                for i in range(start_i, end_i):

                    row = table.row(i)

                    if row.get("incident_state") != "Closed":
                        continue

                    inc = row["number"]

                    if inc in state["incident_seen"]:
                        continue

                    group = get_group(row)
                    made_sla = row.get("made_sla")

                    state["incident_seen"].add(inc)
                    state["total_counts"][group] += 1

                    if made_sla == False:
                        state["breach_counts"][group] += 1

                    breach_rate = (
                        state["breach_counts"][group] /
                        state["total_counts"][group]
                    ) * 100

                    state["breach_rates"][group].append(breach_rate)

                state["cursor"] = end_i

                percent = round((state["cursor"] / table.nrow) * 100, 1)
                remaining = table.nrow - state["cursor"]

                progress_label.value = (
                    f"Processed: {percent}% | Remaining Rows: {remaining}"
                )

                update_plot()
                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

In [7]:
Constructor.widget('Snippet', 2)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

In [11]:
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, Label, DatePicker, VBox, Button
import plotly.graph_objects as go
from datetime import datetime
from collections import defaultdict
import re

@register_snippet
def forensic_priority_boxplot(input_module, input_slot, columns):

    table = input_module.output[input_slot].data()
    out = Output()

    fig = go.FigureWidget()

    start_picker = DatePicker(description="Start Date")
    end_picker = DatePicker(description="End Date")
    run_button = Button(description="Run Analysis", button_style="success")

    progress_label = Label(value="Select date range and click Run.")

    def run_analysis(b):

        progress_label.value = "Processing..."

        durations = defaultdict(list)
        incident_seen = set()

        for i in range(table.nrow):

            row = table.row(i)

            if row.get("incident_state") != "Closed":
                continue

            inc = row["number"]
            if inc in incident_seen:
                continue

            try:
                opened = datetime.strptime(row["opened_at"], "%d/%m/%Y %H:%M")
                closed = datetime.strptime(row["closed_at"], "%d/%m/%Y %H:%M")
            except:
                continue

            # Date filter
            if start_picker.value and opened.date() < start_picker.value:
                continue
            if end_picker.value and opened.date() > end_picker.value:
                continue

            duration = (closed - opened).total_seconds() / 3600

            pr_raw = str(row.get("priority", "3"))
            m = re.search(r"\d+", pr_raw)
            pr = int(m.group(0)) if m else 3

            incident_seen.add(inc)
            durations[pr].append(duration)

        # 🎨 ColorBrewer Sequential Oranges (5-class)
        colors = {
            1: "#a63603",  # Darkest
            2: "#e6550d",
            3: "#fd8d3c",
            4: "#fdbe85",
            5: "#feedde"   # Lightest
        }

        fig.data = []

        for pr in sorted(durations.keys()):
            fig.add_trace(go.Box(
                y=durations[pr],
                name=f"Priority {pr}",
                marker_color=colors.get(pr, "#999999"),
                boxpoints="outliers"
            ))

        fig.update_layout(
            title="Forensic Analysis: Priority vs Resolution Time",
            xaxis_title="Priority Level",
            yaxis_title="Resolution Time (Hours)",
            height=600
        )

        progress_label.value = f"Completed. Incidents analyzed: {len(incident_seen)}"

    run_button.on_click(run_analysis)

    with out:
        display(VBox([start_picker, end_picker, run_button, progress_label]))
        display(fig)

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

## Snippet[3]

In [12]:
Constructor.widget('Snippet', 3)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

In [13]:
import pandas as pd

# Load dataset
df = pd.read_csv("incident_log.csv")

# Convert date columns
df["opened_at"] = pd.to_datetime(df["opened_at"], dayfirst=True, errors="coerce")
df["closed_at"] = pd.to_datetime(df["closed_at"], dayfirst=True, errors="coerce")

# Filter May 2016 (Closed incidents only)
may = df[
    (df["opened_at"] >= "2016-05-01") &
    (df["opened_at"] <= "2016-05-31") &
    (df["incident_state"] == "Closed")
].copy()   

# Extract numeric priority safely
may["priority_num"] = (
    may["priority"]
    .astype(str)
    .str.extract(r"(\d)")
    .astype(float)
)

# Calculate summary statistics
summary = may.groupby("priority_num").agg(
    reassignment_count=("reassignment_count", "median"),
    reopen_count=("reopen_count", "median"),
    sla_breach_rate=("made_sla", lambda x: (x == False).mean()),
    sys_mod_count=("sys_mod_count", "median")
)

print(summary)

              reassignment_count  reopen_count  sla_breach_rate  sys_mod_count
priority_num                                                                  
1.0                          1.0           0.0         0.972973            8.0
2.0                          1.0           0.0         0.990991            6.0
3.0                          0.0           0.0         0.211578            3.0
4.0                          0.0           0.0         0.017316            2.0
